In [1]:
#multiplexing (age based swaps) and parallel fully local policies, swaps done agnostically to outcome of other swaps

In [1]:
import numpy as np
import math
from joblib import Parallel, delayed
import random

In [2]:
def print_state(state,n):
    for i in range(n):
        for j in range(n):
            links=state[i,j]
            print(i,j,':',links)

In [3]:
def sort_state(state,n,n_ch):
    new_state = np.zeros([n,n,n_ch])
    for i in range(n):
        for j in range(n):
            for k in range(n_ch):
                new_state[i,j,k]=state[i,j,k]
        
            new_state[i,j,:] = np.sort(new_state[i,j,:])
    return new_state

def requests(state,n,n_ch,m):
    new_state = np.zeros([n,n,n_ch])
    new_state[:,:,:] = state[:,:,:]
    for i in range(n-1):
        active_links = 0
        for j in range(i+1,n):
            for k in range(n_ch):
                if new_state[i,j,k]!=0:
                    active_links+=1
        
        restartable = []
        for k in range(n_ch):
            if new_state[i,i+1,k]==0:
                restartable.append(k)
                
        for k in range(n_ch-active_links):
            
            if np.random.random()<p_l:
                new_state[i,i+1,restartable[k]]=m
                new_state[i+1,i,restartable[k]]=m
    
    return sort_state(new_state,n,n_ch)  


def waits(state,n,n_ch,m_star):
    new_state = np.zeros([n,n,n_ch])
    new_state[:,:,:] = state[:,:,:]

    for i in range(n-1):
        for j in range(i+1,n):
            for k in range(n_ch):
                if new_state[i,j,k]!=0: # adding 1 to ages of active links or unavailable links
                    new_state[i,j,k]+=1
                    new_state[j,i,k]+=1
                
                if new_state[i,j,k]>m_star: # discarding old links
                    new_state[i,j,k] = 0
                    new_state[j,i,k] = 0
                        
    return sort_state(new_state,n,n_ch)

In [4]:
# i_list=[]
# if n%2==1:
#     for ll in range(1,int(n/2)+1):
#             i_list.append(ll)
#             i_list.append(n-ll)
# else:
#     for ll in range(1,int(n/2)):
#             i_list.append(ll)
#             i_list.append(n-ll)
#     i_list.append(int(n/2))
# i_list = np.array(i_list)-1
def swaps(state, n, n_ch, m_star):
    new_state=np.zeros([n,n,n_ch])
    new_state[:,:,:] =  state[:,:,:]

    for i in range(1,n-1):
        
        active_sites_r=[] #nodes the i-th node (thinks it) is connected to on the right
        active_ages_r=[]
        active_ages_local_r=[] #perceived age of the link to the right
        active_sites_l=[] #nodes the i-th node (thinks it) is connected to on the left
        active_ages_l=[]
        active_ages_local_l=[] #perceived age of the link to the right
        
        #find active or unavailable links to the left and right of the node
        for j in range(i+1,n):
            for k in range(n_ch):
                if new_state[i,j,k]!=0:
                    active_sites_r.append(j)
                    active_ages_r.append(new_state[i,j,k])
                    if new_state[i,j,k]>0:
                        active_ages_local_r.append(new_state[i,j,k])
                    else:
                        active_ages_local_r.append(m_star+new_state[i,j,k])
        
        for j in range(0,i):
            for k in range(n_ch):
                if new_state[i,j,k]!=0:
                    active_sites_l.append(j)
                    active_ages_l.append(new_state[i,j,k])
                    if new_state[i,j,k]>0:
                        active_ages_local_l.append(new_state[i,j,k])
                    else:
                        active_ages_local_l.append(m_star+new_state[i,j,k])
        
        active_sites_l = np.array(active_sites_l)
        active_sites_r = np.array(active_sites_r)
        active_ages_l = np.array(active_ages_l)
        active_ages_r = np.array(active_ages_r)
        active_ages_local_l = np.array(active_ages_local_l)
        active_ages_local_r = np.array(active_ages_local_r)
        
        
        if min(len(active_sites_l),len(active_sites_r))>0: #if at least 1 swap is possible
            
            #sort active links based on ages

            index_l = np.argsort(active_ages_local_l) #sort by ages perceived locally
            index_r = np.argsort(active_ages_local_r) #sort by ages perceived locally
            active_sites_l = active_sites_l[index_l]
            active_ages_l = active_ages_l[index_l]
            active_sites_r = active_sites_r[index_r]
            active_ages_r = active_ages_r[index_r]
            
            for j in range(min(len(active_sites_l),len(active_sites_r))):

                ch_num_r = np.where(new_state[i,active_sites_r[j]]==active_ages_r[j])[0][0] #right channel num to be broken
                ch_num_l = np.where(new_state[active_sites_l[j],i]==active_ages_l[j])[0][0] #left channel num to be broken
                
                if active_ages_l[j]>0 and active_ages_r[j]>0:
                    if np.random.random()<p_sw and (active_ages_r[j] + active_ages_l[j])<=m_star:
                        new_state[active_sites_l[j], active_sites_r[j], 0] = (active_ages_r[j] + active_ages_l[j] - 1)
                        new_state[active_sites_r[j], active_sites_l[j], 0] = (active_ages_r[j] + active_ages_l[j] - 1)
                

#                     if np.absolute(active_sites_r[j]-i)>1 or np.absolute(active_sites_l[j]-i)>1:
                    new_state[i,active_sites_r[j],ch_num_r]=-(m_star-new_state[i, active_sites_r[j], ch_num_r])
                    new_state[active_sites_r[j],i,ch_num_r]=-(m_star-new_state[active_sites_r[j],i,ch_num_r])

#                     else:
#                         new_state[active_sites_r[j],i,ch_num_r]=-1
#                         new_state[i,active_sites_r[j],ch_num_r]=-1
#                         new_state[active_sites_l[j],i,ch_num_l]=-1
#                         new_state[i,active_sites_l[j],ch_num_l]=-1
                
#                     if np.absolute(active_sites_l[j]-i)>1 or np.absolute(active_sites_r[j]-i)>1:
                    new_state[active_sites_l[j],i,ch_num_l]=-(m_star-new_state[active_sites_l[j],i,ch_num_l])
                    new_state[i,active_sites_l[j],ch_num_l]=-(m_star-new_state[i,active_sites_l[j],ch_num_l])
#                     else:
#                         new_state[active_sites_r[j],i,ch_num_r]=-1
#                         new_state[i,active_sites_r[j],ch_num_r]=-1
#                         new_state[active_sites_l[j],i,ch_num_l]=-1
#                         new_state[i,active_sites_l[j],ch_num_l]=-1
                    
                else:
                    if active_ages_r[j]>0 and active_ages_l[j]<0:

#                         if np.absolute(active_sites_r[j]-i)>1 or np.absolute(active_sites_l[j]-i)>1:
                        new_state[i, active_sites_r[j], ch_num_r] = -(m_star-new_state[i, active_sites_r[j], ch_num_r])
                        new_state[active_sites_r[j], i, ch_num_r] = -(m_star-new_state[active_sites_r[j], i, ch_num_r])
#                         else:
#                             new_state[active_sites_r[j],i,ch_num_r]=-1
#                             new_state[i,active_sites_r[j],ch_num_r]=-1
#                             new_state[active_sites_l[j],i,ch_num_l]=-1
#                             new_state[i,active_sites_l[j],ch_num_l]=-1
                            
                    
                    if active_ages_l[j]>0 and active_ages_r[j]<0:

#                         if np.absolute(active_sites_l[j]-i)>1 or np.absolute(active_sites_r[j]-i)>1:
                        new_state[active_sites_l[j], i, ch_num_l] = -(m_star-new_state[active_sites_l[j], i, ch_num_l])
                        new_state[i, active_sites_l[j], ch_num_l] = -(m_star-new_state[i, active_sites_l[j], ch_num_l])
#                         else:
#                             new_state[active_sites_r[j],i,ch_num_r]=-1
#                             new_state[i,active_sites_r[j],ch_num_r]=-1
#                             new_state[active_sites_l[j],i,ch_num_l]=-1
#                             new_state[i,active_sites_l[j],ch_num_l]=-1

                        
#                 new_state=sort_state(new_state,n,n_ch)
        
    return sort_state(new_state,n,n_ch) 

In [5]:
def distill(state,n,n_ch,m_star,level=1):
    new_state=np.zeros([n,n,n_ch])
    new_state[:,:,:] = state[:,:,:]
    
    for i in range(n):
        for j in range(i+1,n):
            non_fresh_channels=[]
            non_fresh_ages=[]
            if np.absolute(i-j)<=level:
                for k in range(n_ch):
                    if new_state[i,j,k]>0:
                        non_fresh_channels.append(k)
                        non_fresh_ages.append(new_state[i,j,k])
                for k in range(len(non_fresh_channels)-1):
                    f1 = 0.25*(1+3*math.exp(-(non_fresh_ages[k]-1)/m_star))
                    f2 = 0.25*(1+3*math.exp(-(non_fresh_ages[k+1]-1)/m_star))
                    p_ds = (8/9)*f1*f2 - (2/9)*(f1+f2) + (5/9)
                    if np.random.random()<p_ds:
                        new_state[i,j,non_fresh_channels[k+1]] = 1+int((m_star/2)*np.log((5-2*(f1+f2)+8*f1*f2)/(12*f1*f2 - 3)))
                        new_state[j,i,non_fresh_channels[k+1]] = 1+int((m_star/2)*np.log((5-2*(f1+f2)+8*f1*f2)/(12*f1*f2 - 3)))
                        new_state[i,j,non_fresh_channels[k]] = 0
                        new_state[j,i,non_fresh_channels[k]] = 0
                    else:
                        new_state[i,j,non_fresh_channels[k]] = 0
                        new_state[j,i,non_fresh_channels[k]] = 0
                        new_state[i,j,non_fresh_channels[k+1]] = 0
                        new_state[j,i,non_fresh_channels[k+1]] = 0
                    
    
    return sort_state(new_state,n,n_ch)

In [6]:
def wt_fid(n,n_ch,m_star,p_l,p_sw,m=1):
    state=np.zeros([n,n,n_ch])
    wt=0
    while len(np.where(state[0][n-1]>0)[0])==0:
        state=requests(state,n,n_ch,m) 
        state=swaps(state,n,n_ch,m_star)
        if len(np.where(state[0][n-1]>0)[0])==0:
                wt+=1
                state=waits(state,n,n_ch,m_star)
        else:
            if (n-1)%2==0:
                wt+=(n-1)/2
                for _ in range(int(n-1/2)):
                    state=waits(state,n,n_ch,m_star)  
            else:
                wt+=(n)/2
                for _ in range(int(n/2)):
                    state=waits(state,n,n_ch,m_star)
#         state=distill(state,n,n_ch,m_star)
   
    return wt,np.min(state[0,n-1][np.where(state[0,n-1]>0)])

def wt_fid_parallel(n,n_ch,m_star,n_ch_ch,p_l,p_sw,m):
    states = []
    for i in range(n_ch_ch):
        states.append(np.zeros([n,n,n_ch]))
    wt=0
    while np.max([states[i][0,n-1,0] for i in range(n_ch_ch)])<=0:
        for i in range(n_ch_ch):
            states[i]=requests(states[i],n,n_ch,m) 
            states[i]=swaps(states[i],n,n_ch,m_star)
            states[i]=waits(states[i],n,n_ch,m_star)
        wt+=1

#             states[i]=distill(states[i],n,n_ch,m_star)
            
                
    ages = np.array([states[i][0,n-1,0] for i in range(n_ch_ch)])
    return wt,np.min(ages[ages>0])

In [ ]:
#multiplexing
wt_all = []
fid_all = []
wt_std = []
fid_std = []
for n in [9]:
    p_l = 0.25
    m_star = 8
    n_ch = 5
    p_sw = 0.5

    wt_wt = []
    fid_fid = []
    print(p_l)
    for k in range(1):
        print(k)
        wt_list = Parallel(n_jobs=100, verbose=50)(delayed(wt_fid)(n,n_ch,m_star,p_l,p_sw) for i in range(200))
        wt_wt.append(np.mean([wt_list[i][0] for i in range(len(wt_list))]))
        fid_fid.append(np.mean([wt_list[i][1] for i in range(len(wt_list))]))

    print(np.mean(wt_wt), np.std(wt_wt), np.mean(fid_fid), np.std(fid_fid))
    wt_all.append(np.mean(wt_wt))
    fid_all.append(np.mean(fid_fid))
    wt_std.append(np.std(wt_wt))
    fid_std.append(np.std(fid_fid))

0.25
0
[Parallel(n_jobs=100)]: Using backend LokyBackend with 100 concurrent workers.


In [10]:
wt_all

[4.88, 43.44, 243.69, 1409.915, 7180.85, 35468.88]

In [11]:
fid_all

[3.29, 3.11, 5.07, 4.065, 7.0, 5.03]